In [ ]:
# Step 3b: Persistent water mask for case-study wetlands

In [ ]:
# ===============================================================
# Step-3 — Persistent water mask (ga_ls_wo_3) + Enc/Ret ablation
# Memory-safe per-wetland pipeline (sequential over 4 wetlands)
# Period: 1988–2023 | Cadence: 5-yr + custom hydrological epochs
# Outputs:
#   1) CSV (BEFORE vs AFTER WOfS transitions)
#   2) Rasters per period:
#        - *_transitions_raw.tif
#        - *_transitions_afterWOfS.tif
#        - *_PersistentWater.tif  (CLIPPED to wetland polygon)
# ===============================================================

import os, gc
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray  # noqa: F401
import datacube
from datacube.utils.geometry import Geometry, CRS
from datacube.utils import masking
from shapely.geometry import mapping
from datetime import datetime
from rasterio.enums import Resampling
from rasterio import features
from numpy import allclose

# -------------------
# CONFIG
# -------------------
START_YEAR   = 1988
END_YEAR     = 2023
INTERVAL_YRS = 5

CUSTOM_EPOCHS = [
    {"label": "1988-1993_WetBaseline",        "start": 1988, "end": 1993},
    {"label": "1994-2000_EarlyDry",           "start": 1994, "end": 2000},
    {"label": "2001-2009_MillenniumDrought",  "start": 2001, "end": 2009},
    {"label": "2010-2012_PostDroughtFloods",  "start": 2010, "end": 2012},
    {"label": "2013-2017_ModerateDry",        "start": 2013, "end": 2017},
    {"label": "2018-2019_SevereDrought",      "start": 2018, "end": 2019},
    {"label": "2020-2022_LaNinaFloods",       "start": 2020, "end": 2022},
]

PIXEL_RES   = 30
PX_TO_HA    = (PIXEL_RES**2) / 10000.0
TARGET_CRS  = "EPSG:3577"
BUFFER_M    = 250  # keep tight to shrink loads

# WOfS persistence settings
WET_FREQ_THRESH = 80.0
MIN_CLEAR_OBS   = 20

BASE_DIR   = "/home/jovyan/DEA products paper/Data"
OUT_TABLES = "/home/jovyan/DEA products paper/Results/step 3_v3_wetland scale/tables"
OUT_RASTERS = "/home/jovyan/DEA products paper/Results/step 3_v3_wetland scale/rasters"
os.makedirs(OUT_TABLES, exist_ok=True)
os.makedirs(OUT_RASTERS, exist_ok=True)

# ---- Four case-study wetlands (update paths as needed) ----
TARGET_WETLANDS = [
    {"name": "Sunshower", "path": "/home/jovyan/DEA products paper/Data/wetland boundaries/Sunshower Lagoon.shp"},
    {"name": "Gooragoon", "path": "/home/jovyan/DEA products paper/Data/wetland boundaries/Gooragoon Lagoon.shp"},
    {"name": "Yarada",    "path": "/home/jovyan/DEA products paper/Data/wetland boundaries/Yarradda Lagoon.shp"},
    {"name": "McKenas",   "path": "/home/jovyan/DEA products paper/Data/wetland boundaries/McKennas Lagoon.shp"},
]

LC_PRODUCT = "ga_ls_landcover_class_cyear_3"

# -------------------
# CLASS GROUPS
# -------------------
WOODY_CODES = {20,27,28,29,30,31, 56,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77}
HERB_CODES  = {21,32,33,34,35,36, 57,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92}
BARE_CODES  = {94,95,96,97}
WATER_CODES = {98,99,100,101,102,103,104}
AMBIG_NAT   = {19,22,23,24,25,26, 55,58,59,60,61,62}
ARTIFICIAL_CODES = {93}
NODATA_CODES     = {255}
CULTIVATED       = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18}
MASK_OUT_CODES   = ARTIFICIAL_CODES | NODATA_CODES | CULTIVATED

# -------------------
# HELPERS
# -------------------
def make_periods(start, end, interval_years=None, custom_epochs=None):
    periods = []
    if interval_years:
        years = list(range(start, end, interval_years))
        for y0 in years:
            y1 = y0 + interval_years
            if y1 <= end:
                periods.append((f"{y0}-{y1}", y0, y1))
    if custom_epochs:
        for ep in custom_epochs:
            label = ep["label"]
            y0, y1 = int(ep["start"]), int(ep["end"])
            if y0 < y1 and (start <= y0 < y1 <= end):
                periods.append((label, y0, y1))
    return periods

def dc_query_from_geom(geom, crs_str=TARGET_CRS, res=PIXEL_RES, buffer_m=BUFFER_M):
    geom_buf = geom.buffer(buffer_m)
    return {
        "geopolygon": Geometry(mapping(geom_buf), CRS(crs_str)),
        "output_crs": crs_str,
        "resolution": (-res, res),
    }

def mask_codes(da: xr.DataArray, keep: set) -> xr.DataArray:
    # dask-safe membership; avoids gufunc loop-dim issues
    return da.astype("int32").isin(list(keep))

def count_px(mask: xr.DataArray) -> int:
    return int(mask.sum().values)

def ensure_singlepart_gdf(path: str, target_crs: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path).to_crs(target_crs)
    if len(gdf) > 1:
        gdf = gpd.GeoDataFrame(geometry=[gdf.unary_union], crs=gdf.crs)
    gdf["WetlandID"] = 1
    return gdf

def write_gtiff(da: xr.DataArray, out_path: str, nodata: int = 0):
    da = da.rio.write_nodata(nodata)
    da.rio.to_raster(
        out_path,
        compress="DEFLATE",
        predictor=2,
        tiled=True,
        blockxsize=512,
        blockysize=512,
        dtype=str(da.dtype)
    )

def align_like(a: xr.DataArray, b: xr.DataArray) -> xr.DataArray:
    """
    Return `a` on the grid of `b` without forcing a reprojection if grids already match.
    Falls back to rioxarray.reproject_match if possible, else interp_like(nearest).
    """
    same_dims = ("y" in a.dims and "x" in a.dims and "y" in b.dims and "x" in b.dims)
    same_shape = same_dims and (a.sizes["y"] == b.sizes["y"] and a.sizes["x"] == b.sizes["x"])
    if same_shape:
        try:
            if a.x.equals(b.x) and a.y.equals(b.y):
                return a
        except Exception:
            if allclose(getattr(a.x, "values", a.x), getattr(b.x, "values", b.x)) and \
               allclose(getattr(a.y, "values", a.y), getattr(b.y, "values", b.y)):
                return a
    # Try reprojection if georeferenced
    try:
        return a.rio.reproject_match(b, resampling=Resampling.nearest)
    except Exception:
        # Coordinate-based nearest alignment as last resort
        return a.interp_like(b, method="nearest")

def build_transition_codes(base_woody: xr.DataArray,
                           tgt_woody: xr.DataArray,
                           keep_mask: xr.DataArray,
                           wetland_mask: xr.DataArray) -> dict:
    base_nonwoody = ~base_woody
    tgt_nonwoody  = ~tgt_woody

    nw_to_w  = base_nonwoody & tgt_woody
    w_to_nw  = base_woody    & tgt_nonwoody
    w_to_w   = base_woody    & tgt_woody
    nw_to_nw = base_nonwoody & tgt_nonwoody

    codes_raw = xr.full_like(base_woody, fill_value=0, dtype="int16")
    inside = wetland_mask.astype(bool)
    codes_raw = xr.where(inside & nw_to_w,  1, codes_raw)  # Encroachment
    codes_raw = xr.where(inside & w_to_nw,  2, codes_raw)  # Retreat
    codes_raw = xr.where(inside & w_to_w,   3, codes_raw)  # Stable Woody
    codes_raw = xr.where(inside & nw_to_nw, 4, codes_raw)  # Stable Non-Woody

    codes_after = xr.where(inside & keep_mask, codes_raw, 0).astype("int16")
    return {"raw": codes_raw, "after": codes_after}

def compute_persistent_water_mask(dc, q, base_like: xr.DataArray):
    """
    Loads WOfS daily, computes wet % and clear counts (chunked),
    returns boolean mask of persistent water aligned to base_like.
    """
    ds_wo = dc.load(
        product="ga_ls_wo_3",
        measurements=["water"],
        time=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
        group_by="solar_day",
        dask_chunks={"time": 64, "x": 1024, "y": 1024},
        **q
    )
    if ds_wo.sizes.get("time", 0) == 0:
        return xr.zeros_like(base_like, dtype=bool)

    water_da = ds_wo["water"]
    wet   = masking.make_mask(water_da, wet=True)
    dry   = masking.make_mask(water_da, dry=True)
    clear = wet | dry

    wet_count   = wet.astype("int16").sum("time")
    clear_count = clear.astype("int16").sum("time")

    with np.errstate(divide="ignore", invalid="ignore"):
        wet_freq_pct = (wet_count.astype("float32") /
                        xr.where(clear_count > 0, clear_count, np.nan)) * 100.0

    # Align to base_like safely
    wf = align_like(wet_freq_pct, base_like)
    cc = align_like(clear_count,  base_like)

    water_mask = (wf > WET_FREQ_THRESH) & (cc >= MIN_CLEAR_OBS)
    water_mask = water_mask.astype(bool).compute()

    # Cleanup big arrays
    del ds_wo, water_da, wet, dry, clear, wet_count, clear_count, wet_freq_pct, wf, cc
    gc.collect()
    return water_mask

# -------------------
# MAIN
# -------------------
periods = make_periods(START_YEAR, END_YEAR, INTERVAL_YRS, CUSTOM_EPOCHS)
print("Total periods (fixed + epochs):", len(periods))

dc = datacube.Datacube(app="WPE_LC_L4_Step3_WOfS_memsafe_clipped")
rows = []

for w in TARGET_WETLANDS:
    name = w["name"]
    print(f"\n▶ Processing wetland: {name}")
    gdf = ensure_singlepart_gdf(w["path"], TARGET_CRS)
    geom = gdf.geometry.iloc[0]

    # Query limited to this wetland + buffer
    q = dc_query_from_geom(geom, buffer_m=BUFFER_M)

    # Load LC once (annual) with chunking
    ds_lc = dc.load(
        product=LC_PRODUCT,
        measurements=["full_classification"],
        time=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
        dask_chunks={"time": -1, "x": 1024, "y": 1024},
        **q
    )
    if ds_lc.sizes.get("time", 0) == 0:
        print("  ! No LC data here; skipping.")
        continue
    lc = ds_lc["full_classification"]
    if not lc.rio.crs:
        lc = lc.rio.write_crs(TARGET_CRS)

    # Rasterize wetland polygon on LC grid (once)
    transform = lc.rio.transform()
    shape = lc.rio.shape
    wmask_np = features.rasterize(
        [(geom, 1)],
        out_shape=shape, transform=transform, fill=0, dtype="uint8"
    )
    wmask = xr.DataArray(wmask_np, dims=("y","x"), coords={"y": lc.y, "x": lc.x}).astype(bool)

    # Base grid to align WOfS
    try:
        base_like = lc.sel(time=lc.time.dt.year == START_YEAR).squeeze(drop=True)
    except Exception:
        base_like = lc.isel(time=0).squeeze(drop=True)
    if not base_like.rio.crs:
        base_like = base_like.rio.write_crs(TARGET_CRS)

    # Persistent water (compute once per wetland)
    water_mask = compute_persistent_water_mask(dc, q, base_like=base_like)
    # Save persistent water **CLIPPED** to wetland polygon
    out_dir = os.path.join(OUT_RASTERS, f"1_{name}")
    os.makedirs(out_dir, exist_ok=True)
    pw_path = os.path.join(out_dir, f"1_{name}_PersistentWater.tif")
    persist_w_clip = xr.where(wmask, water_mask, False).astype("uint8").rio.write_crs(TARGET_CRS)
    write_gtiff(persist_w_clip, pw_path)

    # Count persistent water inside polygon (for CSV)
    persist_px = count_px(persist_w_clip.astype(bool))
    persist_ha = persist_px * PX_TO_HA

    # ---- Period loop for this wetland ----
    for (label, y0, y1) in periods:
        base = lc.sel(time=lc.time.dt.year == y0)
        tgt  = lc.sel(time=lc.time.dt.year == y1)
        if base.sizes.get("time", 0) == 0 or tgt.sizes.get("time", 0) == 0:
            print(f"  · Skip {label}: LC missing {y0} or {y1}")
            continue
        base = base.squeeze(drop=True)
        tgt  = tgt.squeeze(drop=True)
        if not base.rio.crs: base = base.rio.write_crs(TARGET_CRS)
        if not tgt.rio.crs:  tgt  = tgt.rio.write_crs(TARGET_CRS)

        # Validity: natural-only on both years
        valid_base = ~mask_codes(base, MASK_OUT_CODES)
        valid_tgt  = ~mask_codes(tgt,  MASK_OUT_CODES)
        valid = (valid_base & valid_tgt)

        base_woody  = mask_codes(base, WOODY_CODES) & valid
        tgt_woody   = mask_codes(tgt,  WOODY_CODES) & valid

        base_herb   = mask_codes(base, HERB_CODES)  & valid
        base_bare   = mask_codes(base, BARE_CODES)  & valid
        base_water  = mask_codes(base, WATER_CODES) & valid
        base_ambig  = mask_codes(base, AMBIG_NAT)   & valid

        # BEFORE transitions
        enc_h = base_herb  & tgt_woody
        enc_b = base_bare  & tgt_woody
        enc_w = base_water & tgt_woody
        enc_a = base_ambig & tgt_woody
        enc_any = enc_h | enc_b | enc_w | enc_a

        ret_h = mask_codes(base, WOODY_CODES) & mask_codes(tgt, HERB_CODES)  & valid
        ret_b = mask_codes(base, WOODY_CODES) & mask_codes(tgt, BARE_CODES)  & valid
        ret_w = mask_codes(base, WOODY_CODES) & mask_codes(tgt, WATER_CODES) & valid
        ret_a = mask_codes(base, WOODY_CODES) & mask_codes(tgt, AMBIG_NAT)   & valid
        ret_any_clear = ret_h | ret_b | ret_w
        ret_any_ext   = ret_any_clear | ret_a

        # AFTER (~persistent water)
        keep_after = (~water_mask)

        # Build 4-class transition rasters (raw/after) and save
        codes = build_transition_codes(
            base_woody=base_woody, tgt_woody=tgt_woody,
            keep_mask=keep_after, wetland_mask=wmask
        )
        codes_raw   = codes["raw"].rio.write_crs(TARGET_CRS)
        codes_after = codes["after"].rio.write_crs(TARGET_CRS)

        stem = f"1_{name}_{label}"
        write_gtiff(codes_raw,   os.path.join(out_dir, f"{stem}_transitions_raw.tif"))
        write_gtiff(codes_after, os.path.join(out_dir, f"{stem}_transitions_afterWOfS.tif"))

        # ---- Per-wetland counts (BEFORE)
        eH_bef = count_px((enc_h).where(wmask))
        eB_bef = count_px((enc_b).where(wmask))
        eW_bef = count_px((enc_w).where(wmask))
        eA_bef = count_px((enc_a).where(wmask))
        eAny_bef = eH_bef + eB_bef + eW_bef + eA_bef

        rH_bef = count_px((ret_h).where(wmask))
        rB_bef = count_px((ret_b).where(wmask))
        rW_bef = count_px((ret_w).where(wmask))
        rA_bef = count_px((ret_a).where(wmask))
        rAnyClear_bef = rH_bef + rB_bef + rW_bef
        rAnyExt_bef   = rAnyClear_bef + rA_bef

        # AFTER (~persistent water)
        eH_aft = count_px((enc_h & keep_after).where(wmask))
        eB_aft = count_px((enc_b & keep_after).where(wmask))
        eW_aft = count_px((enc_w & keep_after).where(wmask))
        eA_aft = count_px((enc_a & keep_after).where(wmask))
        eAny_aft = eH_aft + eB_aft + eW_aft + eA_aft

        rH_aft = count_px((ret_h & keep_after).where(wmask))
        rB_aft = count_px((ret_b & keep_after).where(wmask))
        rW_aft = count_px((ret_w & keep_after).where(wmask))
        rA_aft = count_px((ret_a & keep_after).where(wmask))
        rAnyClear_aft = rH_aft + rB_aft + rW_aft
        rAnyExt_aft   = rAnyClear_aft + rA_aft

        enc_removed_px = eAny_bef - eAny_aft
        ret_removed_px = rAnyClear_bef - rAnyClear_aft
        enc_removed_ha = enc_removed_px * PX_TO_HA
        ret_removed_ha = ret_removed_px * PX_TO_HA

        rows.append({
            "WetlandName": name, "WetlandID": 1, "SubRegion": "CaseStudy",
            "Period": label, "BaselineYear": y0, "TargetYear": y1,
            "PersistWater_px": persist_px, "PersistWater_ha": persist_ha,
            "Enc_Herb_to_Woody_px_before": eH_bef,
            "Enc_Bare_to_Woody_px_before": eB_bef,
            "Enc_Water_to_Woody_px_before": eW_bef,
            "Enc_Ambig_to_Woody_px_before": eA_bef,
            "Enc_Any_to_Woody_px_before": eAny_bef,
            "Enc_Any_to_Woody_ha_before": eAny_bef * PX_TO_HA,
            "Enc_Herb_to_Woody_px_after": eH_aft,
            "Enc_Bare_to_Woody_px_after": eB_aft,
            "Enc_Water_to_Woody_px_after": eW_aft,
            "Enc_Ambig_to_Woody_px_after": eA_aft,
            "Enc_Any_to_Woody_px_after": eAny_aft,
            "Enc_Any_to_Woody_ha_after": eAny_aft * PX_TO_HA,
            "Ret_Woody_to_Herb_px_before": rH_bef,
            "Ret_Woody_to_Bare_px_before": rB_bef,
            "Ret_Woody_to_Water_px_before": rW_bef,
            "Ret_Any_to_NonWoody_px_before": rAnyClear_bef,
            "Ret_Any_to_NonWoody_ha_before": rAnyClear_bef * PX_TO_HA,
            "Ret_Woody_to_Herb_px_after": rH_aft,
            "Ret_Woody_to_Bare_px_after": rB_aft,
            "Ret_Woody_to_Water_px_after": rW_aft,
            "Ret_Any_to_NonWoody_px_after": rAnyClear_aft,
            "Ret_Any_to_NonWoody_ha_after": rAnyClear_aft * PX_TO_HA,
            "Ret_Woody_to_Ambig_px_before": rA_bef,
            "Ret_Woody_to_Ambig_px_after": rA_aft,
            "Ret_Any_to_NonWoody_Extended_px_before": rAnyExt_bef,
            "Ret_Any_to_NonWoody_Extended_px_after": rAnyExt_aft,
            "Ret_Any_to_NonWoody_Extended_ha_before": rAnyExt_bef * PX_TO_HA,
            "Ret_Any_to_NonWoody_Extended_ha_after": rAnyExt_aft * PX_TO_HA,
            "Enc_Removed_px": enc_removed_px, "Enc_Removed_ha": enc_removed_ha,
            "Ret_Removed_px": ret_removed_px, "Ret_Removed_ha": ret_removed_ha,
        })

    # free per-wetland arrays
    del ds_lc, lc, wmask, water_mask, persist_w_clip
    gc.collect()

# -------------------
# EXPORT TABLE
# -------------------
summary = pd.DataFrame(rows).sort_values(
    ["WetlandName","WetlandID","BaselineYear","TargetYear"]
).reset_index(drop=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(
    OUT_TABLES, f"wofs_mask_{START_YEAR}_{END_YEAR}_{INTERVAL_YRS}yr_plusEpochs_CASESTUDY_{ts}.csv"
)
summary.to_csv(csv_path, index=False)

print("✅ Exported water-mask summary to:", csv_path)
print("🏁 Done. Raster outputs at:", OUT_RASTERS)